# Dataset Preparation for RAG (Retrieval-Augmented Generation)

This notebook provides a complete, end-to-end workflow for preparing a dataset of documents for a **RAG (Retrieval-Augmented Generation)** system. We'll cover:

1. **Document Conversion** with [Docling](https://docling-project.github.io/docling/) - converting PDFs to structured formats
2. **RAG-Optimized Chunking** - breaking documents into semantically meaningful chunks
3. **Embedding Generation** - using Snowflake Arctic embedding model
4. **Vector Storage** - storing embeddings in Milvus vectordb for efficient retrieval

## Why RAG-Specific Preparation Matters

RAG systems require carefully prepared data to ensure:
- **Contextual Completeness**: Each chunk contains sufficient context to be understood independently
- **Semantic Coherence**: Chunks align with natural topic boundaries
- **Optimal Size**: Balanced between detail (larger chunks) and precision (smaller chunks)
- **Metadata Preservation**: Structural information (headings, tables, lists) is retained for better retrieval

## 📦 Installation

Install the required packages. This may take a minute.

In [1]:
%pip install -qq docling docling-core sentence-transformers numpy jupyterlab "pymilvus[milvus_lite]"

Note: you may need to restart the kernel to use updated packages.


## 🔧 Configuration

### Set Input Documents

Define the documents to process. You can mix URLs and local file paths.


In [2]:
# Documents to process - replace with your own
input_files = [
    "https://github.com/docling-project/docling/raw/v2.43.0/tests/data/pdf/2203.01017v2.pdf",
    "https://drive.google.com/file/d/1FO4ymkqK6cT5LM2x87xZvahwp6rmLUPd/view?usp=drive_link",
    "https://drive.google.com/file/d/1rkQ7vDLTekQEIaKd6EyQM5zCaSXGqmNG/view?usp=drive_link",
]

# Output directory for intermediate files
from pathlib import Path
output_dir = Path("data/rag-dataset-preparation/output")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"✓ Will process {len(input_files)} documents")
print(f"✓ Output directory: {output_dir.resolve()}")

✓ Will process 3 documents
✓ Output directory: /Users/roburishabh/Github/odh-data-processing/notebooks/use-cases/data/rag-dataset-preparation/output


## 📄 Step 1: Document Conversion with Docling

### RAG-Optimized Conversion Configuration

For RAG systems, we configure Docling with specific settings:

- **Table Structure Detection**: Essential for preserving tabular data relationships
- **OCR (Optional)**: Enables text extraction from scanned documents
  - **Note**: OCR is disabled by default since most PDFs are digital
  - Set `USE_OCR = True` below if you have scanned documents
  - Requires Tesseract: `brew install tesseract tesseract-lang` (macOS) or `apt-get install tesseract-ocr` (Linux)
- **Image Generation**: Useful for multimodal RAG or image-based context
- **Enrichments** (optional): Code, formulas, picture descriptions add semantic richness

These settings ensure we extract maximum structural and semantic information from documents.


In [3]:
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TesseractOcrOptions
from docling.backend.docling_parse_v4_backend import DoclingParseV4DocumentBackend

# Configure RAG-optimized pipeline
pipeline_options = PdfPipelineOptions()

# Essential settings for RAG
pipeline_options.do_table_structure = True  # Preserve table structure
pipeline_options.table_structure_options.do_cell_matching = True  # Accurate cell matching
pipeline_options.generate_page_images = False  # Set to True if you need images for multimodal RAG
pipeline_options.generate_picture_images = True  # Extract pictures from documents

# OCR configuration - important for scanned documents
# Explicitly use Tesseract OCR engine (avoiding easyocr)
pipeline_options.do_ocr = True
# Uncomment below to force OCR on all pages (useful for scanned documents)
# pipeline_options.ocr_options = TesseractOcrOptions(
#     force_full_page_ocr=False  # Set to True to force OCR on all pages (useful for scanned documents)
# )

# Performance settings
pipeline_options.accelerator_options = AcceleratorOptions(
    num_threads=4,
    device=AcceleratorDevice.AUTO
)

# Optional enrichments - enable these for richer semantic information
# Note: These add processing time but improve RAG quality for technical documents
pipeline_options.do_code_enrichment = False  # Enable for code-heavy documents
pipeline_options.do_formula_enrichment = False  # Enable for math/scientific papers
pipeline_options.do_picture_description = False  # Enable for multimodal RAG
pipeline_options.do_picture_classification = False  # Enable to classify image types

print("✓ Pipeline configured for RAG")

✓ Pipeline configured for RAG


### Run Document Conversion

Convert all documents to Docling's structured format.


In [4]:
import json
from docling_core.types.doc import ImageRefMode

# Create converter
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options,
            backend=DoclingParseV4DocumentBackend,
        )
    }
)

# Store converted documents
converted_documents = []
conversion_results = {}

print("Starting document conversion...\n")

for file_path in input_files:
    print(f"Converting: {file_path}")
    
    result = converter.convert(file_path)
    document = result.document
    
    # Store the document object for later processing
    converted_documents.append({
        'source': file_path,
        'document': document,
        'result': result
    })
    
    # Save intermediate files
    file_name = Path(file_path).stem
    
    # Save JSON (structured format)
    json_path = output_dir / f"{file_name}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(document.export_to_dict(), f, indent=2)
    print(f"  ✓ Saved JSON: {json_path}")
    
    # Save Markdown (human-readable)
    md_path = output_dir / f"{file_name}.md"
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(document.export_to_markdown(image_mode=ImageRefMode.EMBEDDED))
    print(f"  ✓ Saved Markdown: {md_path}")
    
    # Store confidence metrics
    conversion_results[file_path] = {
        'confidence_score': result.confidence.mean_score,
        'confidence_grade': result.confidence.mean_grade.name
    }
    print(f"  ✓ Confidence: {result.confidence.mean_grade.name} (score: {result.confidence.mean_score:.3f})\n")

print(f"✓ Conversion complete! Processed {len(converted_documents)} documents")


Starting document conversion...

Converting: https://github.com/docling-project/docling/raw/v2.43.0/tests/data/pdf/2203.01017v2.pdf


2025-10-11 01:47:57,872 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-11 01:47:57,919 - INFO - Going to convert document batch...
2025-10-11 01:47:57,920 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 55b5e8ff5813a13cd82cbd1a6356f8bd
2025-10-11 01:47:57,928 - INFO - Loading plugin 'docling_defaults'
2025-10-11 01:47:57,929 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-10-11 01:47:57,937 - INFO - Loading plugin 'docling_defaults'
2025-10-11 01:47:57,940 - INFO - Registered ocr engines: ['easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-10-11 01:47:58,060 - INFO - Accelerator device: 'mps'
2025-10-11 01:47:59,966 - INFO - Accelerator device: 'mps'
2025-10-11 01:48:00,978 - INFO - Accelerator device: 'mps'
2025-10-11 01:48:01,342 - INFO - Processing document 2203.01017v2.pdf
2025-10-11 01:48:26,633 - INFO - Finished converting document 2203.01017v2.pdf in 32.44 sec.


  ✓ Saved JSON: data/rag-dataset-preparation/output/2203.01017v2.json
  ✓ Saved Markdown: data/rag-dataset-preparation/output/2203.01017v2.md
  ✓ Confidence: EXCELLENT (score: 0.942)

Converting: https://drive.google.com/file/d/1FO4ymkqK6cT5LM2x87xZvahwp6rmLUPd/view?usp=drive_link


2025-10-11 01:48:27,979 - INFO - detected formats: [<InputFormat.HTML: 'html'>]
2025-10-11 01:48:27,997 - INFO - Going to convert document batch...
2025-10-11 01:48:27,998 - INFO - Initializing pipeline for SimplePipeline with options hash 995a146ad601044538e6a923bea22f4e
2025-10-11 01:48:27,998 - INFO - Processing document uc
2025-10-11 01:48:28,003 - INFO - Finished converting document uc in 1.29 sec.
/Users/roburishabh/Github/odh-data-processing/.venv/lib/python3.9/site-packages/docling/datamodel/base_models.py:441: RuntimeWarning: Mean of empty slice
  np.nanmean(


  ✓ Saved JSON: data/rag-dataset-preparation/output/view?usp=drive_link.json
  ✓ Saved Markdown: data/rag-dataset-preparation/output/view?usp=drive_link.md
  ✓ Confidence: UNSPECIFIED (score: nan)

Converting: https://drive.google.com/file/d/1rkQ7vDLTekQEIaKd6EyQM5zCaSXGqmNG/view?usp=drive_link


2025-10-11 01:48:28,862 - INFO - detected formats: [<InputFormat.HTML: 'html'>]
2025-10-11 01:48:28,885 - INFO - Going to convert document batch...
2025-10-11 01:48:28,886 - INFO - Processing document uc
2025-10-11 01:48:28,890 - INFO - Finished converting document uc in 0.88 sec.


  ✓ Saved JSON: data/rag-dataset-preparation/output/view?usp=drive_link.json
  ✓ Saved Markdown: data/rag-dataset-preparation/output/view?usp=drive_link.md
  ✓ Confidence: UNSPECIFIED (score: nan)

✓ Conversion complete! Processed 3 documents


/Users/roburishabh/Github/odh-data-processing/.venv/lib/python3.9/site-packages/docling/datamodel/base_models.py:441: RuntimeWarning: Mean of empty slice
  np.nanmean(


## 🧩 Step 2: RAG-Optimized Chunking

### Chunking Strategy for RAG

Effective chunking is critical for RAG systems. We'll implement **hybrid chunking** that:

1. **Respects Document Structure**: Uses headings, sections, and natural boundaries
2. **Maintains Context**: Includes metadata like titles, section names
3. **Optimizes Chunk Size**: Tailored to embedding model (1024 dimensions for Snowflake Arctic)
4. **Preserves Relationships**: Keeps tables, lists, and code blocks intact
5. **Smart Merging**: Combines smaller chunks while respecting max token limits

### Docling's Hybrid Chunker

Docling provides a `HybridChunker` that intelligently splits documents while preserving structure and merging smaller chunks for optimal retrieval.

In [5]:
from docling.chunking import HybridChunker
from docling_core.types.doc import DocItemLabel, DoclingDocument
from typing import List, Dict, Any

def chunk_document_for_rag(
    doc: DoclingDocument,
    source: str,
    max_tokens: int = 1024,
    merge_peers: bool = True
) -> List[Dict[str, Any]]:
    """
    Chunk a Docling document using hybrid chunking optimized for RAG.
    
    Args:
        doc: DoclingDocument to chunk
        source: Source file path/URL
        max_tokens: Maximum tokens per chunk (optimized for Snowflake Arctic: 1024)
        merge_peers: Whether to merge smaller chunks at the same level
    
    Returns:
        List of chunks with text and metadata
    """
    # Initialize hybrid chunker with RAG-optimized parameters
    # HybridChunker combines hierarchical structure with smart merging
    chunker = HybridChunker(
        max_tokens=max_tokens,      # Matches Snowflake Arctic's optimal context window
        merge_peers=merge_peers      # Merge small adjacent chunks for better context
    )
    
    # Perform chunking
    chunk_iter = chunker.chunk(doc)
    
    # Process chunks and add metadata
    chunks = []
    for idx, chunk in enumerate(chunk_iter):
        chunk_data = {
            'chunk_id': f"{Path(source).stem}_chunk_{idx}",
            'text': chunk.text,
            'source': source,
            'chunk_index': idx,
            'metadata': {
                'doc_items': [item.label for item in chunk.meta.doc_items] if hasattr(chunk.meta, 'doc_items') else [],
                'headings': chunk.meta.headings if hasattr(chunk.meta, 'headings') else [],
                'token_count': len(chunk.text.split())  # Rough estimate
            }
        }
        chunks.append(chunk_data)
    
    return chunks

print("✓ Chunking function defined")

✓ Chunking function defined


### Apply Chunking to All Documents

In [6]:
# Chunk all documents
all_chunks = []

print("Starting chunking process...\n")

for doc_data in converted_documents:
    source = doc_data['source']
    document = doc_data['document']
    
    print(f"Chunking: {source}")
    
    chunks = chunk_document_for_rag(
        doc=document,
        source=source,
        max_tokens=1024,  # Optimized for Snowflake Arctic embeddings (1024 dims)
        merge_peers=True   # Enable smart merging of smaller chunks
    )
    
    all_chunks.extend(chunks)
    print(f"  ✓ Created {len(chunks)} chunks\n")

print(f"✓ Total chunks created: {len(all_chunks)}")

# Display sample chunk
if all_chunks:
    print("\n=== Sample Chunk ===")
    sample = all_chunks[0]
    print(f"Chunk ID: {sample['chunk_id']}")
    print(f"Source: {sample['source']}")
    print(f"Text preview: {sample['text'][:200]}...")
    print(f"Metadata: {sample['metadata']}")


Starting chunking process...

Chunking: https://github.com/docling-project/docling/raw/v2.43.0/tests/data/pdf/2203.01017v2.pdf


Token indices sequence length is longer than the specified maximum sequence length for this model (531 > 512). Running this sequence through the model will result in indexing errors


  ✓ Created 27 chunks

Chunking: https://drive.google.com/file/d/1FO4ymkqK6cT5LM2x87xZvahwp6rmLUPd/view?usp=drive_link


Token indices sequence length is longer than the specified maximum sequence length for this model (522 > 512). Running this sequence through the model will result in indexing errors


  ✓ Created 1 chunks

Chunking: https://drive.google.com/file/d/1rkQ7vDLTekQEIaKd6EyQM5zCaSXGqmNG/view?usp=drive_link


Token indices sequence length is longer than the specified maximum sequence length for this model (518 > 512). Running this sequence through the model will result in indexing errors


  ✓ Created 1 chunks

✓ Total chunks created: 29

=== Sample Chunk ===
Chunk ID: 2203.01017v2_chunk_0
Source: https://github.com/docling-project/docling/raw/v2.43.0/tests/data/pdf/2203.01017v2.pdf
Text preview: Ahmed Nassar, Nikolaos Livathinos, Maksym Lysak, Peter Staar IBM Research
{ ahn,nli,mly,taa } @zurich.ibm.com...
Metadata: {'doc_items': [<DocItemLabel.TEXT: 'text'>, <DocItemLabel.TEXT: 'text'>], 'headings': ['TableFormer: Table Structure Understanding with Transformers.'], 'token_count': 14}


### Save Chunks to Disk

Save chunks as JSON for inspection and debugging.

In [7]:
chunks_file = output_dir / "rag_chunks.json"
with open(chunks_file, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

print(f"✓ Chunks saved to: {chunks_file.resolve()}")


✓ Chunks saved to: /Users/roburishabh/Github/odh-data-processing/notebooks/use-cases/data/rag-dataset-preparation/output/rag_chunks.json


## 🔢 Step 3: Generate Embeddings with Snowflake Arctic

### About Snowflake Arctic Embeddings

[Snowflake Arctic Embed](https://huggingface.co/Snowflake/snowflake-arctic-embed-l) is a family of text embedding models optimized for retrieval tasks. We'll use the **arctic-embed-l** variant which offers:

- High-quality dense embeddings (1024 dimensions)
- Excellent performance on retrieval benchmarks


In [8]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load Snowflake Arctic embedding model
print("Loading Snowflake Arctic embedding model...")
print("(This may take a few minutes on first run)\n")

embedding_model = SentenceTransformer(
    'Snowflake/snowflake-arctic-embed-l',
    trust_remote_code=True
)

print(f"✓ Model loaded successfully")
print(f"  Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")


2025-10-11 01:48:30,510 - INFO - Use pytorch device_name: mps
2025-10-11 01:48:30,510 - INFO - Load pretrained SentenceTransformer: Snowflake/snowflake-arctic-embed-l


Loading Snowflake Arctic embedding model...
(This may take a few minutes on first run)



2025-10-11 01:48:31,995 - INFO - 1 prompt is loaded, with the key: query


✓ Model loaded successfully
  Embedding dimension: 1024


### Generate Embeddings for All Chunks


In [9]:
def generate_embeddings(chunks: List[Dict[str, Any]], model: SentenceTransformer) -> List[Dict[str, Any]]:
    """
    Generate embeddings for all chunks.
    
    Args:
        chunks: List of chunk dictionaries
        model: SentenceTransformer model
    
    Returns:
        Chunks with added 'embedding' field
    """
    print(f"Generating embeddings for {len(chunks)} chunks...")
    
    # Extract text from chunks
    texts = [chunk['text'] for chunk in chunks]
    
    # Generate embeddings in batches for efficiency
    embeddings = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    
    # Add embeddings to chunks
    for chunk, embedding in zip(chunks, embeddings):
        chunk['embedding'] = embedding.tolist()  # Convert to list for JSON serialization
    
    return chunks

# Generate embeddings
all_chunks = generate_embeddings(all_chunks, embedding_model)

print(f"\n✓ Embeddings generated for all {len(all_chunks)} chunks")
print(f"  Embedding dimension: {len(all_chunks[0]['embedding'])}")


Generating embeddings for 29 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✓ Embeddings generated for all 29 chunks
  Embedding dimension: 1024


## 💾 Step 4: Store in Milvus Vector Database

### About Milvus

[Milvus](https://milvus.io/) is an open-source vector database built for AI applications. It provides:
- Fast similarity search
- Scalable vector storage
- Multiple indexing algorithms
- Hybrid search capabilities

### Using Milvus Lite

We're using **[Milvus Lite](https://milvus.io/docs/milvus_lite.md)** - a lightweight, file-based version of Milvus that:
- Requires no Docker or server setup
- Stores data locally in a file
- Perfect for development, testing, and small-scale deployments
- Uses the same API as full Milvus

For production deployments with larger datasets, consider using the full Milvus server with Docker or Kubernetes.


In [10]:
from pymilvus import (
    connections,
    Collection,
    CollectionSchema,
    FieldSchema,
    DataType,
    utility
)

# Configuration
COLLECTION_NAME = "rag_documents"
EMBEDDING_DIM = len(all_chunks[0]['embedding'])

# Connect to Milvus Lite (file-based, no Docker needed!)
print("Connecting to Milvus Lite (local file-based database)...")
print("This lightweight option is perfect for development and testing.\n")

connections.connect(
    alias="default",
    uri="./milvus_rag_demo.db"  # Creates a local database file
)

print("✓ Connected to Milvus Lite successfully!")
print(f"✓ Database file: ./milvus_rag_demo.db")
print(f"✓ Collection name: '{COLLECTION_NAME}'")
print(f"✓ Embedding dimension: {EMBEDDING_DIM}\n")


/Users/roburishabh/Github/odh-data-processing/.venv/lib/python3.9/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.27.2 is exactly one major version older than the runtime version 6.31.1 at schema.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/Users/roburishabh/Github/odh-data-processing/.venv/lib/python3.9/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.27.2 is exactly one major version older than the runtime version 6.31.1 at common.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/Users/roburishabh/Github/odh-data-processing/.venv/lib/python3.9/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.27.2 is exactly one major version older than the runtime version 6.31.1 at milvus.proto. Please update the gencode to avoid compatibi

Connecting to Milvus Lite (local file-based database)...
This lightweight option is perfect for development and testing.



/Users/roburishabh/Github/odh-data-processing/.venv/lib/python3.9/site-packages/milvus_lite/__init__.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


✓ Connected to Milvus Lite successfully!
✓ Database file: ./milvus_rag_demo.db
✓ Collection name: 'rag_documents'
✓ Embedding dimension: 1024



### Create Collection Schema

Define the schema for our RAG collection with fields for:
- Chunk ID (primary key)
- Text content
- Source document
- Metadata
- Embedding vector


In [11]:
# Define collection schema
fields = [
    FieldSchema(name="id", dtype=DataType.VARCHAR, is_primary=True, max_length=256),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535),
    FieldSchema(name="source", dtype=DataType.VARCHAR, max_length=1024),
    FieldSchema(name="chunk_index", dtype=DataType.INT64),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=EMBEDDING_DIM),
]

schema = CollectionSchema(
    fields=fields,
    description="RAG document chunks with embeddings"
)

print(f"Collection schema defined with {len(fields)} fields")


Collection schema defined with 5 fields


### Create or Recreate Collection


In [12]:
# Drop existing collection if it exists
if utility.has_collection(COLLECTION_NAME):
    print(f"Collection '{COLLECTION_NAME}' exists. Dropping...")
    utility.drop_collection(COLLECTION_NAME)
    print("  ✓ Collection dropped")

# Create new collection
print(f"Creating collection '{COLLECTION_NAME}'...")
collection = Collection(
    name=COLLECTION_NAME,
    schema=schema,
    using='default'
)

print(f"✓ Collection '{COLLECTION_NAME}' created successfully")


Creating collection 'rag_documents'...
✓ Collection 'rag_documents' created successfully


### Insert Data into Milvus


In [13]:
# Prepare data for insertion
print(f"Preparing {len(all_chunks)} chunks for insertion...")

insert_data = [
    [chunk['chunk_id'] for chunk in all_chunks],  # id
    [chunk['text'][:65535] for chunk in all_chunks],  # text (truncate if needed)
    [chunk['source'] for chunk in all_chunks],  # source
    [chunk['chunk_index'] for chunk in all_chunks],  # chunk_index
    [chunk['embedding'] for chunk in all_chunks],  # embedding
]

# Insert data
print("Inserting data into Milvus...")
collection.insert(insert_data)

print(f"✓ Inserted {len(all_chunks)} chunks into Milvus")

# Flush to ensure data is persisted
collection.flush()
print("✓ Data flushed to disk")


Preparing 29 chunks for insertion...
Inserting data into Milvus...
✓ Inserted 29 chunks into Milvus
✓ Data flushed to disk


### Create Index and Load Collection

Create an index on the embedding field to enable search functionality. For small datasets like this demo, a simple **FLAT** index works well. For production with larger datasets (1000+ vectors), consider **IVF_FLAT** or **HNSW** for better performance.


In [14]:
# Define index parameters
# For small datasets (<1000 vectors), FLAT index is simple and fast
# For larger datasets, consider IVF_FLAT or HNSW
index_params = {
    "metric_type": "IP",  # Inner Product (suitable for normalized embeddings)
    "index_type": "FLAT",  # Simple brute-force search, perfect for small datasets
}

# For larger datasets (1000+ vectors), use IVF_FLAT instead:
# index_params = {
#     "metric_type": "IP",
#     "index_type": "IVF_FLAT",
#     "params": {"nlist": 128}  # Number of clusters
# }

print("Creating index on embedding field...")
collection.create_index(
    field_name="embedding",
    index_params=index_params
)

print("✓ Index created successfully")

# Load collection into memory for search
collection.load()
print("✓ Collection loaded into memory")


Creating index on embedding field...
✓ Index created successfully
✓ Collection loaded into memory


## 🔍 Step 5: Test RAG Retrieval

Let's test our RAG system by performing a similarity search.

In [15]:
def search_similar_chunks(query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    """
    Search for similar chunks using semantic similarity.
    
    Args:
        query: Search query text
        top_k: Number of results to return
    
    Returns:
        List of similar chunks with scores
    """
    # Generate embedding for query
    query_embedding = embedding_model.encode([query])[0].tolist()
    
    # Search parameters (FLAT index doesn't require nprobe parameter)
    search_params = {
        "metric_type": "IP",
        "params": {}  # Empty params for FLAT index (would need nprobe for IVF_FLAT)
    }
    
    # Perform search
    results = collection.search(
        data=[query_embedding],
        anns_field="embedding",
        param=search_params,
        limit=top_k,
        output_fields=["id", "text", "source", "chunk_index"]
    )
    
    # Format results
    formatted_results = []
    for hits in results:
        for hit in hits:
            formatted_results.append({
                'chunk_id': hit.entity.get('id'),
                'text': hit.entity.get('text'),
                'source': hit.entity.get('source'),
                'chunk_index': hit.entity.get('chunk_index'),
                'similarity_score': hit.score
            })
    
    return formatted_results

print("✓ Search function defined")

✓ Search function defined


### Perform Multiple Test Searches

Test the RAG system with different queries targeting different documents and topics.

In [16]:
# Define test queries for each document
test_queries = [
    "Provide experimental results and performance metrics of the TableFormer paper?",
    "Give me the summary of First Amendment",
    "Explain cargo theft and provide the statistics of number of states contributed for cargo theft data"
]

# Run each test query and display results
for query_num, query in enumerate(test_queries, 1):
    print(f"\n{'='*80}")
    print(f"🔍 TEST QUERY {query_num}")
    print(f"{'='*80}")
    print(f"Query: '{query}'\n")
    
    # Perform search
    search_results = search_similar_chunks(query, top_k=1)
    
    # Display results
    for idx, result in enumerate(search_results, 1):
        print(f"\n📄 Result {idx}")
        print(f"   Similarity Score: {result['similarity_score']:.4f}")
        print(f"   Source: {result['source'].split('/')[-1][:60]}...")
        print(f"   Chunk ID: {result['chunk_id']}")
        print(f"\n   Text Preview:")
        print(f"   {result['text'][:1000].replace(chr(10), ' ')}...")
        print(f"   {'-'*80}")

print(f"\n{'='*80}")
print(f"✅ RAG System Tested Successfully!")
print(f"✓ Tested {len(test_queries)} queries across different documents")
print(f"{'='*80}")


🔍 TEST QUERY 1
Query: 'Provide experimental results and performance metrics of the TableFormer paper?'



Batches:   0%|          | 0/1 [00:00<?, ?it/s]


📄 Result 1
   Similarity Score: 0.8138
   Source: 2203.01017v2.pdf...
   Chunk ID: 2203.01017v2_chunk_12

   Text Preview:
   TableFormer is evaluated on three major publicly available datasets of different nature to prove the generalization and effectiveness of our model. The datasets used for evaluation are the PubTabNet, FinTabNet and TableBank which stem from the scientific, financial and general domains respectively. We also share our baseline results on the challenging SynthTabNet dataset. Throughout our experiments, the same parameters stated in Sec. 5.1 are utilized....
   --------------------------------------------------------------------------------

🔍 TEST QUERY 2
Query: 'Give me the summary of First Amendment'



Batches:   0%|          | 0/1 [00:00<?, ?it/s]


📄 Result 1
   Similarity Score: 0.6246
   Source: 2203.01017v2.pdf...
   Chunk ID: 2203.01017v2_chunk_2

   Text Preview:
   The occurrence of tables in documents is ubiquitous. They often summarise quantitative or factual data, which is cumbersome to describe in verbose text but nevertheless extremely valuable. Unfortunately, this compact representation is often not easy to parse by machines. There are many implicit conventions used to obtain a compact table representation. For example, tables often have complex columnand row-headers in order to reduce duplicated cell content. Lines of different shapes and sizes are leveraged to separate content or indicate a tree structure. Additionally, tables can also have empty/missing table-entries or multi-row textual table-entries. Fig. 1 shows a table which presents all these issues....
   --------------------------------------------------------------------------------

🔍 TEST QUERY 3
Query: 'Explain cargo theft and provide the statistics of 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


📄 Result 1
   Similarity Score: 0.5724
   Source: 2203.01017v2.pdf...
   Chunk ID: 2203.01017v2_chunk_25

   Text Preview:
   , Sum Sq = 5745.2. , ANOVA. = . , ANOVA.F Value = [266.75. conc, Sum Sq = 2191.39]. conc, ANOVA. = . conc, ANOVA.F Value = . conc, Sum Sq = . conc, ANOVA. = . conc, ANOVA.F Value = 6148. [Residuals, Sum Sq = 236.91. [Residuals, ANOVA. = . [Residuals, ANOVA.F Value =  Figure 10: Example of a complex table with empty cells. Figure 11: Simple table with different style and empty cells. Figure 13: Table predictions example on colorful table. PDF Cells Figure 12: Simple table predictions and post processing. Figure 14: Example with multi-line text....
   --------------------------------------------------------------------------------

✅ RAG System Tested Successfully!
✓ Tested 3 queries across different documents


## 📚 Resources

- [Docling Documentation](https://docling-project.github.io/docling/)
- [Milvus Documentation](https://milvus.io/docs)
- [Snowflake Arctic Embed](https://huggingface.co/Snowflake/snowflake-arctic-embed-l)
- [RAG Best Practices](https://docs.llamaindex.ai/en/stable/optimizing/production_rag/)
- For additional example notebooks related to Data Processing, check the [Open Data Hub Data Processing](https://github.com/opendatahub-io/odh-data-processing/) repository on GitHub. 

## 💬 Feedback

We'd love to hear your feedback! Please [open an issue](https://github.com/opendatahub-io/odh-data-processing/issues) if you have suggestions or run into any problems.


## 🧹 Cleanup (Optional)

Run this cell to clean up resources if needed.


In [17]:
# # Uncomment to drop the collection
# collection.drop()
# print("✓ Collection dropped")

# # Disconnect from Milvus
# connections.disconnect("default")
# print("✓ Disconnected from Milvus")
